# 07. The covariance estimators on the universe

**What this notebook does.** For each of the 306 months of the evaluation window it estimates the covariance matrix, the table of the variances of the members' returns on its diagonal and their covariances off it, from the members' daily returns less the daily risk-free rate, the one-month Treasury bill rate in French's files, over the three years before the month. It does so in four ways, scales each matrix to a month, and records how well conditioned each is. It judges the first clause of expectation E4 on the sample estimator against the Ledoit-Wolf one, and counts the members whose window is not complete and how each is handled.

**Why it comes here.** The mandate of milestone 3 chooses about 500 weights each month by minimising its forecast tracking error, which is computed from one of these matrices. A matrix that understates the risk of some combinations of positions lets the optimiser load on them, so the conditioning is measured before any portfolio is built.

**Terms used in this notebook.**

- *market capitalisation*: price times shares outstanding, at the end of the month before.
- *benchmark*: the universe weighted by market capitalisation at the start of the month.
- *universe*: the stocks in CRSP's list of index members on the last trading day of the month before. A member without a market capitalisation at the start of the month is left out of the benchmark and of the matrices for that month.
- *evaluation window*: July 1999 to December 2024, 306 months, as notebook 05 settled it.
- *excess return*: a return less the risk-free rate.
- *provenance*: the record of what a result was built from, here each French file's address, download time, SHA-256 fingerprint and the line naming the CRSP database it was built from.
- *window*: the trading days of the 36 calendar months before the month, 750 to 759 days.
- *covariance estimator*: a rule that builds a covariance matrix from a window of returns. Each is scaled from a day to a month by multiplying by 21.
- *eigenvector*: a set of weights on the members that makes a portfolio uncorrelated with the portfolios of the other eigenvectors.
- *eigenvalue*: the variance of an eigenvector's portfolio. A covariance matrix has one for each member. A matrix is *positive semi-definite* when none is negative, which every true covariance matrix is, and *singular* when one is zero.
- *condition number*: the largest eigenvalue divided by the smallest. A large one means the matrix is *near-singular*: close to singular, so that solving with it amplifies small errors in its entries.
- *sample estimator*: the covariance of the window's returns, each entry from the data alone. With about 500 members and about 750 days it exists, and its smallest eigenvalues are close to zero.
- *shrinkage*: pulling an estimate towards a simple target, called the *shrinkage target*. The *Ledoit-Wolf estimator* pulls the sample matrix towards a scaled identity matrix, the average variance on the diagonal and zero elsewhere, by a weight from zero to one called the *shrinkage intensity*, chosen from the data to minimise the expected error (Ledoit and Wolf 2004).
- *factor*: a common source of return.
- *loading*: the slope of a member's return on a factor in a regression. A *beta* is the same thing under its usual name.
- *factor model*: each member's return explained by its loadings on French's five factors and his momentum factor. It gives the covariance as the loadings times the factors' covariance times the loadings, plus each member's own variance, called *idiosyncratic*, the variance of what the factors leave.
- *principal component*: one of the eigenvectors of a covariance matrix, ranked by its eigenvalue. The *PCA estimator* takes the benchmark's excess return as the first factor, with each member's beta on it, describes what is left by the first four principal components of the members' residual covariance, and adds each member's own remaining variance.
- *complete member*: a member that lacks a return on at most five days of the window. The sample, Ledoit-Wolf and PCA estimators need a matrix with no gap, so they are estimated on the complete members, on the days on which all of them have a return. A day on which a complete member lacks a return is left out of the window. At most 21 days may be left out: where more would go, the bar for complete falls from five days to four and so on, and the month is counted.
- *model row*: the row of the matrix for a member that is not complete. It comes from a model fitted on the member's own days: its loadings on the six factors and its idiosyncratic variance for the sample, Ledoit-Wolf and factor estimators, and its beta on the benchmark and its residual variance for the PCA estimator. A member with fewer than 60 returns, too few for seven coefficients, takes the market-capitalisation-weighted mean loadings and the median residual variance of the complete members of its GICS sector, when that sector has at least ten of them, and of the whole universe otherwise. The *degrees of freedom* of a regression are its observations less its coefficients.
- *assembled matrix*: the matrix the mandate will use, complete members and model rows together. Each estimator has one construction for every month, positive semi-definite by construction, and its block for the complete members is what the copied function returns for them. The rules are in `src/slt/constants.py`, section 7.
- *expectation*: a test fixed in advance in `src/slt/constants.py`, reported as met or not met in the same words either way. Its result is the *verdict*.

**What I expect to see, and where it comes from.**

- *E4, first clause*: in every month of the evaluation window, the condition number of the sample estimator at least ten times that of the Ledoit-Wolf estimator, on the assembled matrices. About 500 members on about 750 days leave the smallest sample eigenvalues near zero, and shrinkage lifts them. The second clause, on realised tracking error, is judged in milestone 3. The condition numbers of the complete members' blocks are printed beside them with no verdict.
- *The members*: from structural counts made on 11 November 2026, before this notebook was written, about 490 complete members a month and 478 at the least, and about 11 model rows a month and 22 at the most, of which about 3 have fewer than 252 returns and 8 at the most. Fewer than 60 returns: 190 member-months in 134 months.
- *The check*: every assembled matrix is positive semi-definite by construction, so the notebook computes each one's smallest eigenvalue and stops if one is below minus one ten-billionth of the largest. That would be a fault in the code, and nothing from such a run is reported.
- *Share classes*, a check with no verdict, fixed on 12 November 2026 once the results of section 4 were known: from structural counts, the universe holds one company with two share classes from July 1999 to 2004, none from 2005 to 2013, and up to five from 2014. Where both classes are complete members their daily returns are correlated at about 0.98, which sets the sample matrix's smallest eigenvalue. Section 5 counts them and recomputes the ratio with one class for each company.

Running time: about ten to twenty minutes, most of it the 306 months of four estimators on about 500 members.

In [ ]:
import json
import pathlib
import sys
import time

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from slt import constants as C
from slt import extraction as X

root = X.data_root()             # stops with one sentence when the data root is not set
manifest = X.check_manifest(root)   # stops when the manifest on disk is not the one the constants fix
print("manifest SHA-256", C.MANIFEST_SHA256[:12], "| files", manifest["counts"]["files"],
      "| rows", f"{manifest['counts']['rows']:,}")

In [ ]:
import numpy as np

from slt import estimators as E
from slt import french as FR
from slt import layer as L
from slt import universe as U
from slt import validation as V

## 1. The members, their market capitalisations and sectors, and French's daily factors

The universe, its monthly panel and its sectors are rebuilt from the files as in notebooks 02, 03 and 05, for the months from July 1996, the first month of the first window, to December 2024. French's daily factors and risk-free rate are downloaded once into `data/french/`.

In [ ]:
t0 = time.time()
universe = L.load_universe(root)
data = L.load_panel(root, universe)
panel = data["panel"]
first_window = pd.Period(C.EVAL_START, "M") - 12 * C.COV_DAILY_WINDOW_YEARS
p = panel[panel["month"] >= first_window]
sectors = L.load_sectors(root, p)
members = sectors["classified"][["month", "permno", "sector"]].merge(p[["month", "permno", "cap"]], on=["month", "permno"])
names, _ = X.read_table(root, "crsp_msenames")
company = U.companies(universe["ulong"][universe["ulong"]["month"] >= first_window], names)
members = members.merge(company[["month", "permno", "permco"]], on=["month", "permno"], how="left")   # held in memory only
factors, rf, records = FR.daily_factors()
for r in records:
    print(r["filename"], "|", r["vintage_line"], "| SHA-256", r["zip_sha256"][:12])
print(f"{time.time() - t0:.0f} s | member-months from {first_window}: {len(members):,} | without a capitalisation: {members['cap'].isna().sum():,}")

## 2. The daily excess returns and the benchmark's

The daily stock file is read a year at a time from 1996 for the stocks that are members in any month from July 1996, and held as one matrix in memory. The benchmark's daily excess return is each day's return of that month's members weighted by their market capitalisations at the start of the month, rescaled over the members with a return that day, less the risk-free rate. It is the PCA estimator's first factor.

In [ ]:
t0 = time.time()
years = [y for y in L.daily_years() if y >= first_window.year]
daily, calendar = L.daily_matrix(root, members["permno"].unique(), years, progress=True)
rf_d = rf.reindex(daily.index)
excess = daily.sub(rf_d, axis=0)
bench = E.benchmark_daily_excess(daily, rf, members)
del daily
missing_factor_days = int(factors.reindex(excess.index).isna().any(axis=1).sum())
print(f"{time.time() - t0:.0f} s | trading days {len(excess.index):,} | stocks {excess.shape[1]:,} | days without French's factors: {missing_factor_days}")

## 3. The four estimators, month by month

For each month the four assembled matrices are built, checked and measured, and only their scalars are kept: no matrix and no member's row leaves the loop. A year's months are printed as they finish.

In [ ]:
t0 = time.time()
factors_d = factors.reindex(excess.index)
rows, pairs = [], []
for month in pd.period_range(C.EVAL_START, C.EVAL_END, freq="M"):
    m = members[(members["month"] == month) & members["cap"].notna()]
    res = E.month_estimates(month, excess, factors_d, bench, m)
    res.pop("matrices")
    pairs += [{"month": month, "correlation": r} for r in res.pop("pair_correlations")]
    E.check_positive_semidefinite(res)
    rows.append(res)
    if month.month == 12:
        print(month.year, f"{time.time() - t0:.0f} s", end=" | ", flush=True)
rows = pd.DataFrame(rows)
print()
print(f"months {len(rows)} | every assembled matrix positive semi-definite within the tolerance")

## 4. Conditioning by year, and E4's first clause

The table by year gives each estimator's condition number (median, least and largest), the complete members' median beside it, the smallest ratio of the sample's condition number to the Ledoit-Wolf one's and the months below ten, the shrinkage intensity, and the counts of complete members, model rows and days left out. Every fifth year is printed, and the table is committed in full.

In [ ]:
table = E.by_year(rows)
shown = ["year", "months"] + [f"condition_{n}_median" for n in C.COV_ESTIMATORS] + [
    "ratio_sample_to_ledoit_wolf_least", "months_ratio_below_level", "lw_intensity_mean", "complete_members_mean",
    "model_rows_mean", "model_rows_under_252_mean", "model_rows_under_60_member_months", "days_left_out_most", "months_bar_fell"]
print(table.loc[table["year"] % 5 == 0, shown].round(2).to_string(index=False))
e4 = E.judge_e4_first_clause(rows)
print(V.verdict_line("E4, first clause", e4["met"],
      f"The sample's condition number is at least {C.E4_CONDITION_RATIO_MIN:.0f} times the Ledoit-Wolf one's in "
      f"{e4['months_at_or_above_level']} of {e4['months']} months, the smallest ratio {e4['smallest_ratio']:.1f} in "
      f"{e4['month_of_smallest_ratio']}, the median {e4['median_ratio']:.1f}"))
counts = {"complete_members_mean": float(rows["complete"].mean()), "complete_members_least": int(rows["complete"].min()),
          "model_rows_mean": float(rows["model_rows"].mean()), "model_rows_most": int(rows["model_rows"].max()),
          "model_rows_under_252_mean": float(rows["model_rows_under_252"].mean()),
          "model_rows_under_252_most": int(rows["model_rows_under_252"].max()),
          "model_rows_under_60_member_months": int(rows["model_rows_under_60"].sum()),
          "months_with_a_model_row_under_60": int((rows["model_rows_under_60"] > 0).sum()),
          "days_left_out_most": int(rows["days_left_out"].max()), "months_bar_fell": int(rows["bar_fell"].sum()),
          "days_without_factors": int(rows["days_without_factors"].sum())}
print(json.dumps(counts, indent=1))

## 5. A check on share classes, with no verdict

This check was fixed on 12 November 2026, once the results of section 4 were known, to explain the months in which the ratio reaches E4's level. It carries no verdict, and E4's level and verdict do not move. Some companies have more than one share class in the index, each a member with its own PERMNO, for example a voting and a non-voting class. CRSP gives each company a permanent company number, the PERMCO, which the names file holds for every security over time, read here on the same day as membership. Two classes of one company can move almost together: the difference between them is then a portfolio with almost no variance, and it sets the smallest eigenvalue of the sample matrix. The check counts, by month, the companies with more than one class among the members and the pairs of complete members that are classes of one company, with the correlation of their daily returns over the window. It then keeps one class for each company, the one with the larger market capitalisation at the start of the month, and recomputes the sample and Ledoit-Wolf condition numbers on the complete members of that universe by the same rules. The pairs and their correlations are held in memory: only counts, medians and the months at the level are written, and no weight of these companies, which before 2005 would be one company's weight.

In [ ]:
pairs = pd.DataFrame(pairs, columns=["month", "correlation"])
check = E.share_class_check(rows, pairs)
print(f"companies with more than one class among the members: {check['companies_with_more_than_one_class_mean']:.2f} a month on average, "
      f"{check['companies_with_more_than_one_class_most']} at the most | classes left out with one for each company: {check['classes_left_out_mean']:.2f} a month")
median_corr = check["median_correlation_from_that_month"]
print(f"pairs of complete members that are classes of one company, by month: {check['share_class_pairs_by_month']}, "
      f"of which {check['share_class_pairs_from_that_month']} from {check['share_class_pairs_from']}, median correlation "
      + ("none, with no pair" if median_corr is None else f"{median_corr:.3f}"))
print(f"with one class for each company, the ratio is at or above {C.E4_CONDITION_RATIO_MIN:.0f} in {check['months_one_class_at_or_above_level']} of {len(rows)} months "
      f"{check['months_one_class_at_or_above_level_by_year']}, median {check['ratio_one_class_median']:.2f}, least {check['ratio_one_class_least']:.2f}, "
      f"largest {check['ratio_one_class_largest']:.1f}; no verdict")

## 6. What is committed

Four files go into `results/20260928/`, each an aggregate with no identifier of a security and no series by month: the conditioning and the counts by year, with the check on share classes by year, a summary with E4's first clause, the eigenvalue check and the check on share classes over the window, the verdicts of milestone 2 with E4's first clause added beside E3, and the provenance record of French's files.

In [ ]:
summary = {
    "written_by": "notebooks/07_covariance.ipynb", "manifest_sha256": C.MANIFEST_SHA256,
    "french_vintages": {r["key"]: r["vintage_line"] for r in records},
    "estimators": list(C.COV_ESTIMATORS), "window_years": C.COV_DAILY_WINDOW_YEARS, "scale": C.DAILY_TO_MONTHLY_SCALE,
    "pca_components": C.PCA_N_COMPONENTS, "complete_window_missing_max": C.COMPLETE_WINDOW_MISSING_MAX,
    "complete_days_left_out_max": C.COMPLETE_DAYS_LEFT_OUT_MAX, "model_row_own_min": C.FACTOR_ROW_OWN_MIN,
    "sector_min_complete": C.SECTOR_MIN_COMPLETE, "sample_monthly": C.SAMPLE_MONTHLY_STATUS,
    "e4_first_clause": e4, "read_on": C.E4_READ_ON, "share_class_check": check,
    "eigenvalue_check": {"tolerance": C.PSD_FAULT_TOLERANCE, "passed_every_month": True,
                         "smallest_ratio_by_estimator": {n: float(rows[f"smallest_eigenvalue_ratio_{n}"].min()) for n in C.COV_ESTIMATORS}},
    "counts": counts,
}
path = ROOT / C.RESULTS_DIR / "verdicts_milestone2.json"
record = json.loads(path.read_text()) if path.exists() else {}
record["E4_first_clause"] = {k: e4[k] for k in ("met", "months", "months_at_or_above_level", "smallest_ratio", "month_of_smallest_ratio", "median_ratio")}
written = [X.write_results("covariance_by_year.csv", frame=table),
           X.write_results("covariance_summary.json", record=summary),
           X.write_results("verdicts_milestone2.json", record=record),
           FR.write_provenance(records)]
for p in written:
    print("written:", p.relative_to(ROOT))

## What was learned, and what could be wrong

What the notebook establishes: how close to singular each of the four estimators is, month by month over the evaluation window, whether the sample estimator's condition number is at least ten times the Ledoit-Wolf one's in every month, judged as E4's first clause, and how many members each month needed a model row.

What could still be wrong. A condition number says how much a matrix amplifies errors. Whether its forecasts are right is for the second clause of E4 and for the bias statistic, the ratio of realised to forecast risk, both in milestone 3. A model row rests on the member's own days, and for a recent listing those can be few and unrepresentative. The factor model's rows use French's factors, built from all US stocks, for a universe of large ones. And the window weights every day of three years equally, so a matrix lags a change in how stocks move together.